# 06 Quality Control

Validate raster readability, CRS, dimensions, alignment, non-empty masks, and write reports.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from igcd.config import load_config
from igcd.quality_control import (
    validate_alignment,
    validate_nonempty_mask,
    validate_raster,
    write_quality_reports,
)

config = load_config(PROJECT_ROOT / 'config' / 'config.json')
records = []
for path in sorted(config.paths['processed'].glob('**/*.tif')):
    records.extend(validate_raster(path))
for mask_path in sorted((config.paths['processed'] / 'masks').glob('*/*_mask.tif')):
    records.append(validate_nonempty_mask(mask_path, config.raw['quality_control']['min_mask_pixels']))
    glacier_id = '_'.join(mask_path.stem.split('_')[:2])
    year = mask_path.parent.name
    image = config.paths['exports'] / year / f'{glacier_id}_{year}_sentinel.tif'
    if image.exists():
        records.append(validate_alignment(image, mask_path))

write_quality_reports(
    records,
    config.paths['reports'] / 'quality_report.csv',
    config.paths['reports'] / 'quality_summary.json'
)